# Serverless validity validation — gbx_st_makevalid + gbx_st_explainvalidity

Proves the light-tier validity functions work end-to-end on real Serverless (e2-demo / oauth-fe).

**Key check (orientation fix):** `gbx_st_makevalid` default mode applies `orient_polygons` after
`make_valid` so its output passes the product `ST_IsValid` — which enforces ring orientation
(same-winding outer+hole = invalid) while Shapely/GEOS ignores orientation. This cannot be
verified in Docker (no product ST_ functions available locally).

**Four stages:**
1. `explainvalidity` correctness — JSON shape + valid/code/location per invalidity class
2. `makevalid` default passes product `ST_IsValid`; `ogc` opt-in leaves orientation unfixed
3. Product-vs-Shapely coverage matrix — confirms orientation divergence (spike §3)
4. Benchmark — `explainvalidity` + `makevalid` throughput; A/B extended-parse overhead

In [ ]:
import datetime
import json
import time

N_PART = 8
N_BENCH = 2000

# ---------------------------------------------------------------------------
# Corpus — invalidity classes from the spike + valid controls.
# Columns: id, label, wkt, shapely_valid (GEOS verdict), expect_code (code when invalid)
#
# id=7 (same_winding_hole): Shapely/GEOS says VALID (ignores ring orientation);
#   the product's ST_IsValid says INVALID (enforces hole-winding-opposite-exterior).
#   This is the orientation divergence the spike documented (spike row 10).
# ---------------------------------------------------------------------------
CORPUS = [
    # id  label                         wkt                                                                                        shapely_valid  code
    (1,  "bowtie_self_intersection",    "POLYGON((0 0,1 1,1 0,0 1,0 0))",                                                        False,          10),
    (2,  "hole_outside_shell",          "POLYGON((0 0,0 3,3 3,3 0,0 0),(5 5,5 6,6 5,5 5))",                                     False,          20),
    (3,  "nested_holes",                "POLYGON((0 0,0 10,10 10,10 0,0 0),(1 1,1 9,9 9,9 1,1 1),(2 2,2 8,8 8,8 2,2 2))",       False,          21),
    (4,  "ring_self_intersection",      "POLYGON((0 0,2 2,4 0,4 4,2 2,0 4,0 0))",                                               False,          11),
    (5,  "disconnected_interior",       "POLYGON((0 0,0 10,10 10,10 0,0 0),(0 5,5 10,10 5,5 0,0 5))",                           False,          22),
    (6,  "valid_control",               "POLYGON((0 0,0 1,1 1,1 0,0 0))",                                                       True,           0),
    (7,  "same_winding_hole",           "POLYGON((0 0,0 10,10 10,10 0,0 0),(2 2,2 4,4 4,4 2,2 2))",                             True,           0),
    # ^ outer CCW, hole CCW (both same winding) — OGC requires hole be CW.
    # ^ Shapely: valid (GEOS ignores orientation); product ST_IsValid: INVALID.
]

results = {
    "run_date": datetime.datetime.utcnow().isoformat() + "Z",
    "n_corpus": len(CORPUS),
    "stages": {},
}

print(f"Corpus: {len(CORPUS)} geometries")
for row in CORPUS:
    print(f"  {row[0]:2d}. {row[1]:30s}  shapely_valid={row[3]}  code={row[4]}")

In [ ]:
# Register the light VectorX validity SQL functions from the staged wheel.
from databricks.labs.gbx.pyvx import functions as pyvx_functions

pyvx_functions.register(spark, only=["gbx_st_explainvalidity", "gbx_st_makevalid"])
print("Registered: gbx_st_explainvalidity, gbx_st_makevalid")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 1: explainvalidity correctness
# Assert JSON shape + valid/reason/code/location match expected per class.
# ---------------------------------------------------------------------------
schema = "id INT, label STRING, wkt STRING, expect_valid BOOLEAN, expect_code INT"
corpus_rows = [(c[0], c[1], c[2], c[3], c[4]) for c in CORPUS]
df_corpus = spark.createDataFrame(corpus_rows, schema)

df_explain = df_corpus.repartition(N_PART, "id").selectExpr(
    "id", "label", "expect_valid", "expect_code",
    "gbx_st_explainvalidity(wkt) AS detail_json",
)
explain_rows = df_explain.orderBy("id").collect()

print("explainvalidity results:")
bad = []
for r in explain_rows:
    d = json.loads(r["detail_json"])
    # Validate JSON shape: required keys always present
    for k in ("valid", "reason", "code", "location"):
        if k not in d:
            bad.append(f"  id={r['id']} missing key '{k}'")
    if d["valid"] != r["expect_valid"]:
        bad.append(f"  id={r['id']} {r['label']}: valid={d['valid']} expected={r['expect_valid']}")
    if d["code"] != r["expect_code"]:
        bad.append(f"  id={r['id']} {r['label']}: code={d['code']} expected={r['expect_code']}")
    if not r["expect_valid"] and r["id"] not in (6, 7):  # invalid geoms should have location when GEOS embeds it
        if d["location"] is not None and not d["location"].startswith("POINT("):
            bad.append(f"  id={r['id']} location not a POINT WKT: {d['location']}")
    print(f"  {r['id']:2d} {r['label']:30s}  valid={d['valid']}  code={d['code']}  location={d['location']}")

assert not bad, "explainvalidity correctness failures:\n" + "\n".join(bad)
results["stages"]["explain_correctness"] = {
    "n": len(explain_rows), "pass": True,
}
print(f"\nPASS: explainvalidity returned correct valid/code for all {len(explain_rows)} corpus rows")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2: makevalid default mode passes product ST_IsValid
# This is the KEY on-cluster check — proves the orient_polygons fix works.
# ST_IsValid is a DBR built-in (unavailable in Docker; only verifiable here).
# ---------------------------------------------------------------------------
df_all = spark.createDataFrame(
    [(c[0], c[1], c[2]) for c in CORPUS], "id INT, label STRING, wkt STRING"
)

df_repaired = (
    df_all.repartition(N_PART, "id")
    .selectExpr(
        "id", "label",
        "gbx_st_makevalid(wkt) AS repaired_wkb",           # default: linework + orient_polygons
        "gbx_st_makevalid(wkt, 'ogc') AS repaired_ogc",   # ogc opt-in: topology only, no orient
    )
)

# Apply product ST_IsValid gate to the repaired geometries
df_gate = df_repaired.selectExpr(
    "id", "label",
    "repaired_wkb IS NOT NULL AS not_null",
    "ST_IsValid(ST_GeomFromEWKB(repaired_wkb)) AS product_passes_default",
    "ST_IsValid(ST_GeomFromEWKB(repaired_ogc)) AS product_passes_ogc",
)

gate_rows = df_gate.orderBy("id").collect()

print("makevalid default + product ST_IsValid gate:")
for r in gate_rows:
    print(
        f"  {r['id']:2d} {r['label']:30s}  not_null={r['not_null']}"
        f"  product_default={r['product_passes_default']}"
        f"  product_ogc={r['product_passes_ogc']}"
    )

# ALL makevalid-default outputs must pass the product gate
fails = [r for r in gate_rows if not r["product_passes_default"] or not r["not_null"]]
assert not fails, (
    f"makevalid default output failed product ST_IsValid: "
    f"{[(r['id'], r['label']) for r in fails]}"
)

# The same_winding_hole (id=7) with ogc opt-in should NOT pass the product gate
# (orientation unfixed), documenting the expected ogc-vs-product divergence
same_winding_row = next((r for r in gate_rows if r["label"] == "same_winding_hole"), None)
ogc_leaves_orientation_unfixed = (
    same_winding_row is not None and not same_winding_row["product_passes_ogc"]
)
print(f"\nogc opt-in leaves same-winding orientation unfixed (product still rejects): {ogc_leaves_orientation_unfixed}")

results["stages"]["makevalid_product_gate"] = {
    "n": len(gate_rows),
    "default_pass": True,
    "ogc_leaves_orientation_unfixed": ogc_leaves_orientation_unfixed,
}
print(f"\nPASS: all {len(gate_rows)} makevalid-default outputs pass product ST_IsValid")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: Product-vs-Shapely coverage matrix
# Extends the spike: for each corpus row, compare product ST_IsValid vs
# Shapely's verdict (via gbx_st_explainvalidity.valid) — a table confirming
# agreement on topology + the orientation divergence.
# ---------------------------------------------------------------------------
df_cov = spark.createDataFrame(
    [(c[0], c[1], c[2]) for c in CORPUS], "id INT, label STRING, wkt STRING"
)

df_matrix = df_cov.repartition(N_PART, "id").selectExpr(
    "id", "label",
    "ST_IsValid(ST_GeomFromText(wkt)) AS product_is_valid",   # product verdict
    "gbx_st_explainvalidity(wkt) AS detail_json",              # Shapely verdict + detail
)

matrix_rows = df_matrix.orderBy("id").collect()

print(f"{'id':>3}  {'label':30s}  {'shapely':8}  {'product':8}  {'agree':5}  reason")
print("-" * 100)
matrix = []
for r in matrix_rows:
    d = json.loads(r["detail_json"])
    shapely_v = d["valid"]        # Shapely/GEOS verdict
    product_v = r["product_is_valid"]
    agree = shapely_v == product_v
    matrix.append({
        "id": r["id"], "label": r["label"],
        "shapely_valid": shapely_v, "product_valid": product_v,
        "agree": agree, "reason": d["reason"],
    })
    mark = "OK" if agree else "DIVERGE"
    print(
        f"  {r['id']:2d}  {r['label']:30s}  {str(shapely_v):8}  {str(product_v):8}  {mark:7}"
        f"  {d['reason'][:55]}"
    )

diverge = [m for m in matrix if not m["agree"]]
print(f"\nAgreements: {len(matrix) - len(diverge)}/{len(matrix)}  Divergences: {len(diverge)}")
for dv in diverge:
    print(f"  DIVERGE id={dv['id']} {dv['label']}: shapely={dv['shapely_valid']} product={dv['product_valid']}")

# Orientation divergence: Shapely=valid, product=invalid (same-winding ring)
orientation_divergences = [
    m for m in diverge if m["shapely_valid"] and not m["product_valid"]
]
orientation_divergence_confirmed = len(orientation_divergences) > 0
print(f"\nOrientation divergence confirmed (Shapely valid, product invalid): {orientation_divergence_confirmed}")
assert orientation_divergence_confirmed, (
    "Expected at least one orientation divergence (same-winding-hole case). "
    "Spike finding not reproduced on this Serverless env."
)

results["stages"]["coverage_matrix"] = {
    "n_rows": len(matrix),
    "divergences": len(diverge),
    "orientation_divergence_confirmed": orientation_divergence_confirmed,
    "diverge_labels": [m["label"] for m in diverge],
    "matrix": matrix,
}
print(f"PASS: coverage matrix shows {len(diverge)} divergence(s) — orientation gap confirmed")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 4: Benchmark
# Throughput of explainvalidity + makevalid over a synthetic corpus.
# A/B: core-only (bare is_valid_reason, no JSON/regex) vs full explainvalidity
# to confirm the extended-parsing overhead is marginal vs GEOS + UDF boundary.
# ---------------------------------------------------------------------------
from pyspark.sql.types import StringType

# Register a core-only UDF: just is_valid_reason, no JSON encoding / regex extraction
def _explain_core_only(geom):
    """Minimal validity check: raw GEOS reason string only (no JSON, no code, no location)."""
    from shapely import is_valid_reason
    from shapely.io import from_wkb, from_wkt
    try:
        if isinstance(geom, (bytes, bytearray, memoryview)):
            g = from_wkb(bytes(geom))
        elif isinstance(geom, str):
            g = from_wkt(geom)
        else:
            return None
        return is_valid_reason(g)
    except Exception:
        return None

spark.udf.register("_explain_core_only", _explain_core_only, StringType())

# Synthetic corpus: 2000 rows, cycling bowtie / valid / hole-outside
bench_wkts = []
for i in range(N_BENCH):
    if i % 3 == 0:
        bench_wkts.append((i, "POLYGON((0 0,1 1,1 0,0 1,0 0))"))       # bowtie
    elif i % 3 == 1:
        bench_wkts.append((i, "POLYGON((0 0,0 1,1 1,1 0,0 0))"))       # valid
    else:
        bench_wkts.append((i, "POLYGON((0 0,0 3,3 3,3 0,0 0),(5 5,5 6,6 5,5 5))"))  # hole outside

df_bench = (
    spark.createDataFrame(bench_wkts, "id INT, wkt STRING")
    .repartition(N_PART, "id")
    .cache()
)
# Materialize cache to exclude DataFrame construction from timing
df_bench.count()
print(f"Benchmark corpus: {N_BENCH} rows, {N_PART} partitions (cached)")

# A: core-only (raw reason string)
t0 = time.time()
n_core = df_bench.selectExpr("_explain_core_only(wkt) AS reason").count()
t_core = time.time() - t0

# B: full explainvalidity (JSON + regex + code mapping)
t0 = time.time()
n_full = df_bench.selectExpr("gbx_st_explainvalidity(wkt) AS detail").count()
t_full = time.time() - t0

# C: makevalid throughput (default mode)
t0 = time.time()
n_makevalid = df_bench.selectExpr("gbx_st_makevalid(wkt) AS repaired").count()
t_makevalid = time.time() - t0

overhead_pct = (t_full - t_core) / t_core * 100 if t_core > 0 else 0.0

print(f"\nBenchmark ({N_BENCH} rows, {N_PART} partitions):")
print(f"  explainvalidity core-only  : {t_core:.2f}s  ({N_BENCH / t_core:.0f} rows/s)")
print(f"  explainvalidity full       : {t_full:.2f}s  ({N_BENCH / t_full:.0f} rows/s)")
print(f"  makevalid (default)        : {t_makevalid:.2f}s  ({N_BENCH / t_makevalid:.0f} rows/s)")
print(f"  extended overhead          : {overhead_pct:+.1f}%")

# Extended parsing overhead should be marginal (< 50% of core — UDF boundary dominates)
assert abs(overhead_pct) < 50, (
    f"Extended-parsing overhead unexpectedly large: {overhead_pct:.1f}%. "
    "JSON/regex overhead should not dominate vs GEOS + UDF boundary cost."
)

results["stages"]["benchmark"] = {
    "n_bench": N_BENCH,
    "n_partitions": N_PART,
    "t_core_only_sec": round(t_core, 3),
    "t_full_sec": round(t_full, 3),
    "t_makevalid_sec": round(t_makevalid, 3),
    "rows_per_sec_core": round(N_BENCH / t_core, 1),
    "rows_per_sec_full": round(N_BENCH / t_full, 1),
    "rows_per_sec_makevalid": round(N_BENCH / t_makevalid, 1),
    "extended_overhead_pct": round(overhead_pct, 1),
    "overhead_marginal": abs(overhead_pct) < 50,
}
print("PASS: extended-parsing overhead is marginal vs GEOS + UDF boundary")

In [ ]:
# ---------------------------------------------------------------------------
# Final: assemble results, assert overall pass, exit with structured output.
# ---------------------------------------------------------------------------
results["pass"] = all([
    results["stages"].get("explain_correctness", {}).get("pass"),
    results["stages"].get("makevalid_product_gate", {}).get("default_pass"),
    results["stages"].get("coverage_matrix", {}).get("orientation_divergence_confirmed"),
    results["stages"].get("benchmark", {}).get("overhead_marginal"),
])

print(json.dumps(results, indent=2))

if results["pass"]:
    print("\nPASS: all validity validation checks green.")
    print("  - explainvalidity: correct valid/code/location for all corpus classes")
    print("  - makevalid default: all outputs pass product ST_IsValid (orientation fix verified)")
    print("  - coverage matrix: orientation divergence confirmed (Shapely valid, product invalid)")
    overhead = results["stages"]["benchmark"]["extended_overhead_pct"]
    print(f"  - benchmark: extended overhead {overhead:+.1f}% vs core-only (marginal)")
else:
    print("\nFAILED: see results dict above for details.")

assert results["pass"], "Validity validation FAILED — see results dict for details."
dbutils.notebook.exit(json.dumps(results))